# Operator actions and Symbolica export

This notebook walks through the **compiled Lagrangian** layer:

- **`Model` → `model.lagrangian()`** gives a `CompiledLagrangian` of ordered `InteractionTerm`s (the structure the vertex code trusts).
- **`Model.apply_operator(...)`** and **`L.apply_operator(...)`** act on that compiled layer directly.
- **`FieldOperator`** 
- **`TermOperator`** 
- **`partial(mu)`** 
- **`L.ibp_normal_form()`** 
- **`L.to_symbolica()`** 



In [1]:
from dataclasses import replace
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for p in (ROOT, SRC):
    s = str(p)
    if s not in sys.path:
        sys.path.insert(0, s)

from symbolica import Expression, S

from model import Model, PartialD, dirac_field, flavor_index, scalar_field
from lagrangian.operator_action import (
    FieldOperator,
    OperatorAtomResult,
    OperatorExpansionError,
    OperatorSummand,
    TermOperator,
    partial,
    replacement_operator,
    single_field_result,
)


## 1. Define fields and build the model

In [2]:
def field_factor(occ):
    name = occ.field.name
    if occ.conjugated and not occ.field.self_conjugate:
        name = name + ".bar"
    return name


def describe_term(t, index):
    factors = " · ".join(field_factor(o) for o in t.fields)
    if t.derivatives:
        d = ", ".join(f"slot {a.target} → ∂_{a.lorentz_index}" for a in t.derivatives)
    else:
        d = "(none)"
    bil = t.closed_dirac_bilinears or "(none)"
    print(f"[{index}] {factors}")
    print(f"     ∂: {d}   bilinears: {bil}")



In [3]:
Generation = flavor_index("Generation", 3, prefix="f")
mu = S("mu")
lam = S("lam")
f = S("f")


Chi = scalar_field("Chi", self_conjugate=True)
Phi = scalar_field("Phi", self_conjugate=True)
psi = dirac_field("psi", indices=())
eta = dirac_field("eta", indices=())

lepton = dirac_field(
    "l",
    class_members=("e", "mu", "ta"),
    indices=(Generation,),
    flavor_index=Generation,
)
H = scalar_field("H", self_conjugate=True)



In [4]:

model_phi3 = Model(Phi * Phi * Phi + Phi * PartialD(Phi, mu))
L_phi3 = model_phi3.lagrangian()

## 3. Symbolica export (`to_symbolica`)


In [5]:

expr = L_phi3.to_symbolica()
print("exported expression:")
print(expr)


print("\nPderivative wrt  Phi:")
print(expr.derivative(S("Phi")))



exported expression:
Phi*PartialD(Phi,mu)+Phi^3

Pderivative wrt  Phi:
Phi*der(1,0,PartialD(Phi,mu))+PartialD(Phi,mu)+3*Phi^2


In [6]:
expr_coord = L_phi3.to_symbolica(derivative_style="coordinate")
print("coordinate-dependent export of L:")
print(expr_coord)
print()
print("spacetime-style derivative wrt x_mu:")
print(expr_coord.derivative(S("x_mu")))


coordinate-dependent export of L:
der(1,Phi(x_mu))*Phi(x_mu)+Phi(x_mu)^3

spacetime-style derivative wrt x_mu:
3*der(1,Phi(x_mu))*Phi(x_mu)^2+der(2,Phi(x_mu))*Phi(x_mu)+der(1,Phi(x_mu))^2


## 4. `apply_operator`


###  `Phi -> Chi`.

In [7]:
replace_phi = replacement_operator("Phi_to_Chi", {Phi: Chi()})
L_replaced_model = model_phi3.apply_operator(replace_phi)
L_replaced = L_phi3.apply_operator(replace_phi)

print("Simple replacement Phi -> Chi")
print("Number of output terms:", len(L_replaced.terms))
for i, term in enumerate(L_replaced.terms):
    describe_term(term, i)

print("Symbolica view of O[L]:", L_replaced.to_symbolica())
print("feynman rules view:", L_replaced.feynman_rule())


Simple replacement Phi -> Chi
Number of output terms: 5
[0] Chi · Phi · Phi
     ∂: (none)   bilinears: (none)
[1] Phi · Chi · Phi
     ∂: (none)   bilinears: (none)
[2] Phi · Phi · Chi
     ∂: (none)   bilinears: (none)
[3] Chi · Phi
     ∂: slot 1 → ∂_mu   bilinears: (none)
[4] Phi · Chi
     ∂: slot 1 → ∂_mu   bilinears: (none)
Symbolica view of O[L]: Chi*PartialD(Phi,mu)+3*Chi*Phi^2+Phi*PartialD(Chi,mu)
feynman rules view: {('Chi', 'Phi', 'Phi'): 6𝑖, ('Chi', 'Phi'): pcomp(q1,mu1_int)+pcomp(q2,mu1_int)}



### `Phi -> a*Chi + b*Omega`.


In [8]:
Omega = scalar_field("Omega", self_conjugate=True)

phi_to_linear_combo = FieldOperator(
    "phi_to_linear_combo",
    on_field=lambda occ: (
        None
        if occ.field is not Phi
        else OperatorAtomResult(
            summands=(
                OperatorSummand(coefficient=S("a"), replacement=(Chi(),)),
                OperatorSummand(coefficient=S("b"), replacement=(Omega(),)),
            )
        )
    ),
)

L_field_linear = model_phi3.apply_operator(phi_to_linear_combo)
print("One acted Phi slot can become a*Chi or b*Omega.")
print(L_field_linear.to_symbolica())
print("=====================================================================")
for i, term in enumerate(L_field_linear.terms):
    describe_term(term, i)

One acted Phi slot can become a*Chi or b*Omega.
Chi*a*PartialD(Phi,mu)+3*Chi*Phi^2*a+Phi*a*PartialD(Chi,mu)+Phi*b*PartialD(Omega,mu)+Omega*b*PartialD(Phi,mu)+3*Phi^2*Omega*b
[0] Chi · Phi · Phi
     ∂: (none)   bilinears: (none)
[1] Omega · Phi · Phi
     ∂: (none)   bilinears: (none)
[2] Phi · Chi · Phi
     ∂: (none)   bilinears: (none)
[3] Phi · Omega · Phi
     ∂: (none)   bilinears: (none)
[4] Phi · Phi · Chi
     ∂: (none)   bilinears: (none)
[5] Phi · Phi · Omega
     ∂: (none)   bilinears: (none)
[6] Chi · Phi
     ∂: slot 1 → ∂_mu   bilinears: (none)
[7] Omega · Phi
     ∂: slot 1 → ∂_mu   bilinears: (none)
[8] Phi · Chi
     ∂: slot 1 → ∂_mu   bilinears: (none)
[9] Phi · Omega
     ∂: slot 1 → ∂_mu   bilinears: (none)


In [9]:

prepend_omega_per_phi = FieldOperator(
    "prepend_omega_per_phi",
    on_field=lambda occ: (
        None if occ.field is not Phi else single_field_result((Omega(), occ))
    ),
)

prepend_omega_once = TermOperator(
    "prepend_omega_once",
    apply_to_term=lambda term: (replace(term, fields=(Omega(),) + term.fields),),
)

L_slotwise = L_phi3.apply_operator(prepend_omega_per_phi)
L_termwise = L_phi3.apply_operator(prepend_omega_once)

print("FieldOperator: Omega is inserted once for each acted Phi slot.")
print(L_slotwise.to_symbolica())
for i, term in enumerate(L_slotwise.terms):
    describe_term(term, i)
print("=====================================================================")
print("TermOperator: Omega is inserted once per InteractionTerm.")
print(L_termwise.to_symbolica())
for i, term in enumerate(L_termwise.terms):
    describe_term(term, i)



FieldOperator: Omega is inserted once for each acted Phi slot.
2*Phi*Omega*PartialD(Phi,mu)+Phi^2*PartialD(Omega,mu)+3*Phi^3*Omega
[0] Omega · Phi · Phi · Phi
     ∂: (none)   bilinears: (none)
[1] Phi · Omega · Phi · Phi
     ∂: (none)   bilinears: (none)
[2] Phi · Phi · Omega · Phi
     ∂: (none)   bilinears: (none)
[3] Omega · Phi · Phi
     ∂: slot 2 → ∂_mu   bilinears: (none)
[4] Phi · Omega · Phi
     ∂: slot 1 → ∂_mu   bilinears: (none)
[5] Phi · Omega · Phi
     ∂: slot 2 → ∂_mu   bilinears: (none)
TermOperator: Omega is inserted once per InteractionTerm.
Phi*Omega*PartialD(Phi,mu)+Phi^3*Omega
[0] Omega · Phi · Phi · Phi
     ∂: (none)   bilinears: (none)
[1] Omega · Phi · Phi
     ∂: slot 1 → ∂_mu   bilinears: (none)


### 4.2 Composition, algebra, and expansion guards

**`apply_operators(...)`** applies operators left-to-right. **`operator_bracket(...)`** and **`operator_anticommutator(...)`** expose the graded algebra on compiled terms. **`OperatorExpansionError`** is the fail-closed guard when a slot-wise replacement plus derivative Leibniz would branch too much.


In [10]:
term_scale = TermOperator(
    "term_scale",
    apply_to_term=lambda term: (replace(term, coupling=2 * term.coupling),),
)

replace_phi = replacement_operator("Phi_to_Chi", {Phi: Chi()})
split_phi = replacement_operator(
    "split_phi",
    {Phi: single_field_result((Chi(), Phi()))},
)

L_term_chain = L_phi3.apply_operators(replace_phi, term_scale)

print("apply_operators(replace_phi, term_scale):", L_term_chain.to_symbolica())
print("graded bracket [term_scale, prepend_omega_once]:", L_phi3.operator_bracket(term_scale, prepend_omega_once).to_symbolica())

try:
    L_phi3.apply_operator(split_phi, max_generated_terms=1)
except OperatorExpansionError as exc:
    print("Expansion cap:", exc)


apply_operators(replace_phi, term_scale): 2*Chi*PartialD(Phi,mu)+6*Chi*Phi^2+2*Phi*PartialD(Chi,mu)
graded bracket [term_scale, prepend_omega_once]: 0
Expansion cap: Operator expansion exceeds configured limit (operator='split_phi', slot=1, replacement_len=2, derivative_count_on_slot=0, projected_terms=1, max_generated_terms=1).


## 5. Flavor-expanded export (`flavor_expand`)

 **`apply_operator(..., flavor_expand=True)`** 

In [11]:
flavor_model = Model(S("g") * lepton.bar(f) * lepton(f) * H)
L_fl = flavor_model.lagrangian()

compact = L_fl.to_symbolica()
expanded = L_fl.to_symbolica(flavor_expand=True)

print("Generic (class field 'l'):")
print(compact)
print()
print("Flavor-expanded (e, mu, ta):")
print(expanded)

Generic (class field 'l'):
H*g*l(f,i_decl_1)*lbar(f,i_decl_1)

Flavor-expanded (e, mu, ta):
H*g*mu(i_decl_1)*mubar(i_decl_1)+H*g*ebar(i_decl_1)*e(i_decl_1)+H*g*tabar(i_decl_1)*ta(i_decl_1)


## 7. Scalar total-derivative investigation

Goal: investigate the minimal scalar identity

$$\partial_\mu(\phi^2\,\partial_\mu\phi)
= 2\,\phi\,(\partial_\mu\phi)(\partial_\mu\phi) + \phi^2\,\Box\phi$$

and therefore, **only modulo a total derivative / boundary term under the spacetime integral**, 

$$\phi^2\,\Box\phi \simeq -2\,\phi\,(\partial_\mu\phi)(\partial_\mu\phi).$$

In [12]:
c = S("c")
phi = Phi

current_model = Model(
    phi * phi * PartialD(phi, mu),
)

expanded_divergence_model = Model(
    (
        2 * phi * PartialD(phi, mu) * PartialD(phi, mu)
        + phi * phi * PartialD(PartialD(phi, mu), mu)
    ),
)

L1_model = Model(
    c * phi * phi * PartialD(PartialD(phi, mu), mu),
)

L2_model = Model(
    -2 * c * phi * PartialD(phi, mu) * PartialD(phi, mu),
)

delta_model = Model(
    (
        c * phi * phi * PartialD(PartialD(phi, mu), mu)
        + 2 * c * phi * PartialD(phi, mu) * PartialD(phi, mu)
    ),
)

J = current_model.lagrangian()
J_expanded = expanded_divergence_model.lagrangian()
L1_td = L1_model.lagrangian()
L2_td = L2_model.lagrangian()
Delta_td = delta_model.lagrangian()

In [13]:
expr_J_expanded = J_expanded.to_symbolica()
expr_L1 = L1_td.to_symbolica()
expr_L2 = L2_td.to_symbolica()
expr_Delta = Delta_td.to_symbolica()

NF_L1 = L1_td.ibp_normal_form()
NF_L2 = L2_td.ibp_normal_form()
NF_Delta = Delta_td.ibp_normal_form()

print("L1 =")
print(expr_L1)
print()
print("L2 =")
print(expr_L2)
print()
print("L1 - L2 =")
print(expr_Delta)
print()
print("IBP normal form of L1:")
print(NF_L1.to_symbolica())
print()
print("IBP normal form of L2:")
print(NF_L2.to_symbolica())
print()
print("IBP normal form of L1 - L2:")
print(NF_Delta.to_symbolica())

assert NF_Delta.terms == ()
print()
print("Check passed: L1 - L2 vanishes in the scalar IBP normal form.")


L1 =
Phi^2*c*PartialD(PartialD(Phi,mu),mu)

L2 =
-2*Phi*c*PartialD(Phi,mu)^2

L1 - L2 =
2*Phi*c*PartialD(Phi,mu)^2+Phi^2*c*PartialD(PartialD(Phi,mu),mu)

IBP normal form of L1:
2/3*Phi*c*PartialD(Phi,mu)^2+4/3*Phi^2*c*PartialD(PartialD(Phi,mu),mu)

IBP normal form of L2:
2/3*Phi*c*PartialD(Phi,mu)^2+4/3*Phi^2*c*PartialD(PartialD(Phi,mu),mu)

IBP normal form of L1 - L2:
0

Check passed: L1 - L2 vanishes in the scalar IBP normal form.


## 8. Spacetime derivative operator: `partial(mu)`

`partial(mu)`creates fresh `DerivativeAction`s on the acted slots.




In [14]:
L_partial = L_phi3.apply_operator(partial(mu))

print("Source:           ", L_phi3.to_symbolica())
print("partial(mu)[L]:  ", L_partial.to_symbolica())
print()
for i, term in enumerate(L_partial.terms):
    describe_term(term, i)

print("======================================================================")
print("Feynman rules view:", L_partial.feynman_rule(Phi,Phi,Phi, include_delta=True))

Source:            Phi*PartialD(Phi,mu)+Phi^3
partial(mu)[L]:   Phi*PartialD(PartialD(Phi,mu),mu)+3*Phi^2*PartialD(Phi,mu)+PartialD(Phi,mu)^2

[0] Phi · Phi · Phi
     ∂: slot 0 → ∂_mu   bilinears: (none)
[1] Phi · Phi · Phi
     ∂: slot 1 → ∂_mu   bilinears: (none)
[2] Phi · Phi · Phi
     ∂: slot 2 → ∂_mu   bilinears: (none)
[3] Phi · Phi
     ∂: slot 1 → ∂_mu, slot 0 → ∂_mu   bilinears: (none)
[4] Phi · Phi
     ∂: slot 1 → ∂_mu, slot 1 → ∂_mu   bilinears: (none)
Feynman rules view: 6*(2*𝜋)^d*pcomp(q1,mu1_int)*Delta(q1+q2+q3)+6*(2*𝜋)^d*pcomp(q2,mu1_int)*Delta(q1+q2+q3)+6*(2*𝜋)^d*pcomp(q3,mu1_int)*Delta(q1+q2+q3)


### 8.1 partial(mu) on Phi * Chi

In [15]:
product_model = Model(Phi * Chi).lagrangian()
print("partial(mu)[Phi * Chi] =", product_model.apply_operator(partial(mu)).to_symbolica())

restricted = product_model.apply_operator(partial(mu, on=Phi))
print("partial(mu, on=Phi)[Phi * Chi] =", restricted.to_symbolica())

try:
    product_model.apply_operator(partial(mu), max_generated_terms=1)
except OperatorExpansionError as exc:
    print("capped fan-out:", exc)


partial(mu)[Phi * Chi] = Chi*PartialD(Phi,mu)+Phi*PartialD(Chi,mu)
partial(mu, on=Phi)[Phi * Chi] = Chi*PartialD(Phi,mu)
capped fan-out: Operator expansion exceeds configured limit (operator='d_mu', slot=1, replacement_len=1, derivative_count_on_slot=0, projected_terms=1, max_generated_terms=1).


## 9. SU(3) × SU(2) × U(1) export check

In [16]:
from model import GaugeFixing, build_unbroken_standard_model

sm = build_unbroken_standard_model()
xiB = S("xiB")
xiW = S("xiW")
xiG = S("xiG")

sm_decl_with_gf = (
    sm.lagrangians.LSM
    + GaugeFixing(sm.gauge_groups.U1Y, xi=xiB)
    + GaugeFixing(sm.gauge_groups.SU2L, xi=xiW)
    + GaugeFixing(sm.gauge_groups.SU3C, xi=xiG)
)

sm_model_with_gf = Model(
    lagrangian_decl= sm_decl_with_gf,
    name="SM-unbroken-with-gf",
    gauge_groups=(sm.gauge_groups.U1Y, sm.gauge_groups.SU2L, sm.gauge_groups.SU3C),
    parameters=sm.model.parameters,
)
L_sm_gf = sm_model_with_gf.lagrangian()
expr_sm_gf = L_sm_gf.to_symbolica()

print("Full SM with gauge fixing, Symbolica export:",expr_sm_gf)
expr_sm_gf_text = str(expr_sm_gf)

print("Gauge groups:", [group.name for group in sm_model_with_gf.gauge_groups])
print("Source declarations:", len(sm_decl_with_gf.source_terms))
print("Lowered terms:", len(L_sm_gf.terms))
print("Symbolica string length:", len(expr_sm_gf_text))
print()
print("Preview:")
print(expr_sm_gf_text[:1500] + " ...")


Full SM with gauge fixing, Symbolica export: -lam*Phi0(w_decl_1)*Phi0(w_decl_2)*Phibar0(w_decl_1)*Phibar0(w_decl_2)+1/2*g1*g2*g(mink(4, mu),mink(4, nu))*t(coad(3, weak_adj_Wi_SU2L_mix),cof(2, w_bar_Phi),cof(2, w_Phi))*Phi0(w_Phi)*Phibar0(w_bar_Phi)*B0(mu)*Wi0(nu,weak_adj_Wi_SU2L_mix)+1/2*g1*g2*g(mink(4, mu),mink(4, nu))*t(coad(3, weak_adj_Wi_SU2L_mix),cof(2, w_bar_Phi),cof(2, w_Phi))*Phi0(w_Phi)*Phibar0(w_bar_Phi)*B0(nu)*Wi0(mu,weak_adj_Wi_SU2L_mix)+1/6*g1*g(cof(2, w_bar_qL_weak_fund_id),cof(2, w_qL_weak_fund_id))*g(cof(3, fl_bar_qL_generation_id),cof(3, fl_qL_generation_id))*g(cof(3, c_bar_qL_color_fund_id),cof(3, c_qL_color_fund_id))*gamma(bis(4, i_bar_qL_U1Y),bis(4, i_qL_U1Y),mink(4, mu))*qL0(i_qL_U1Y,w_qL_weak_fund_id,fl_qL_generation_id,c_qL_color_fund_id)*qLbar0(i_bar_qL_U1Y,w_bar_qL_weak_fund_id,fl_bar_qL_generation_id,c_bar_qL_color_fund_id)*B0(mu)-1/2*g1*g(cof(2, w_bar_lL_weak_fund_id),cof(2, w_lL_weak_fund_id))*g(cof(3, fl_bar_lL_generation_id),cof(3, fl_lL_generation_id))*ga

## 10. Replacement-style operator checks



In [17]:
from lagrangian.operator_action import OperatorAtomResult, OperatorSummand

g = S("g")
c = S("c")


### 10.0 Minimal odd-sign example


In [18]:
fermion_model = Model(
    psi.bar() * psi() * psi.bar() * psi(),
)
L_f = fermion_model.lagrangian()


def odd_s_on_psi(occurrence):
    if occurrence.field is not psi:
        return None
    return single_field_result(eta.occurrence(conjugated=occurrence.conjugated))


s_odd = FieldOperator(name="s", parity=1, on_field=odd_s_on_psi)
L_s = L_f.apply_operator(s_odd)

print("Original:", len(L_f.terms), "term(s)")
for t in L_f.terms:
    print("  coupling", t.coupling, "→", " · ".join(field_factor(o) for o in t.fields))

print("\nAfter odd s on each psi slot:", len(L_s.terms), "terms")
print("O[L_f]=", L_s.to_symbolica())
for t in L_s.terms:
    print("  coupling", t.coupling, "→", " · ".join(field_factor(o) for o in t.fields))


Original: 1 term(s)
  coupling 1 → psi.bar · psi · psi.bar · psi

After odd s on each psi slot: 4 terms
O[L_f]= -psi(i_decl_1)*psibar(i_decl_1)*psibar(i_decl_2)*eta(i_decl_2)+psi(i_decl_1)*psi(i_decl_2)*psibar(i_decl_1)*etabar(i_decl_2)+psi(i_decl_1)*psi(i_decl_2)*psibar(i_decl_2)*etabar(i_decl_1)-psi(i_decl_2)*psibar(i_decl_1)*psibar(i_decl_2)*eta(i_decl_1)
  coupling 1 → eta.bar · psi · psi.bar · psi
  coupling -1 → psi.bar · eta · psi.bar · psi
  coupling 1 → psi.bar · psi · eta.bar · psi
  coupling -1 → psi.bar · psi · psi.bar · eta


### 10.1 Odd-operator signs

Test the graded Leibniz signs on $L = g\,\bar\psi\,\psi\,\phi$ with an odd operator $s$.


In [19]:
psi = dirac_field("psi", indices=())
xi = dirac_field("xi", indices=())
phi = scalar_field("phi", self_conjugate=True)
chi= scalar_field("chi", self_conjugate=True)

odd_model = Model(
    g * psi.bar() * psi() * phi* phi* phi,
)
L_odd_slots = odd_model.lagrangian()


def s_on_field_slot(occ):
    if occ.field is psi:
        return single_field_result((xi(conjugated=occ.conjugated),))
    if occ.field is phi:
        return single_field_result((chi(),))
    return None


s_slot = FieldOperator(name="s", parity=1, on_field=s_on_field_slot)
L_odd_out = L_odd_slots.apply_operator(s_slot)

print("Source:", odd_model.lagrangian_decl.source_terms[0])
print()
for i, term in enumerate(L_odd_out.terms):
    print("coupling", term.coupling)
    describe_term(term, i)
print()
print("symbolica preview:", L_odd_slots.to_symbolica())
print("Symbolica export:")
print(L_odd_out.to_symbolica())

print()
print("Feynman rules for (xi.bar, psi, phi, phi, phi):", L_odd_out.feynman_rule(xi.bar,psi,phi,phi,phi))
print("Feynman rules for (psi.bar, xi, phi, phi, phi):", L_odd_out.feynman_rule(psi.bar,xi,phi,phi,phi))
print("Feynman rules for (psi.bar, psi, chi, phi, phi):", L_odd_out.feynman_rule(psi.bar,psi,chi,phi,phi))


Source: g * psi.bar * psi * phi * phi * phi

coupling g
[0] xi.bar · psi · phi · phi · phi
     ∂: (none)   bilinears: ((0, 1),)
coupling -g
[1] psi.bar · xi · phi · phi · phi
     ∂: (none)   bilinears: ((0, 1),)
coupling g
[2] psi.bar · psi · chi · phi · phi
     ∂: (none)   bilinears: ((0, 1),)
coupling g
[3] psi.bar · psi · phi · chi · phi
     ∂: (none)   bilinears: ((0, 1),)
coupling g
[4] psi.bar · psi · phi · phi · chi
     ∂: (none)   bilinears: ((0, 1),)

symbolica preview: phi^3*g*psi(i_decl_1)*psibar(i_decl_1)
Symbolica export:
3*phi^2*g*chi*psi(i_decl_1)*psibar(i_decl_1)+phi^3*g*psi(i_decl_1)*xibar(i_decl_1)-phi^3*g*psibar(i_decl_1)*xi(i_decl_1)

Feynman rules for (xi.bar, psi, phi, phi, phi): 6𝑖*g*g(bis(4, i1),bis(4, i2))
Feynman rules for (psi.bar, xi, phi, phi, phi): -6𝑖*g*g(bis(4, i1),bis(4, i2))
Feynman rules for (psi.bar, psi, chi, phi, phi): 6𝑖*g*g(bis(4, i1),bis(4, i2))


### 10.2 Multiple summands per slot



In [20]:
phi_sum = scalar_field("phi_sum", self_conjugate=True)
eta_sum = scalar_field("eta_sum", self_conjugate=True)
A_sum = scalar_field("A_sum", self_conjugate=True)
B_sum = scalar_field("B_sum", self_conjugate=True)
R_sum = scalar_field("R_sum", self_conjugate=True)

sum_model = Model(
    g * phi_sum * eta_sum,
)
L_sum_slots = sum_model.lagrangian()


def O_sum_slot(occ):
    if occ.field is phi_sum:
        return OperatorAtomResult(
            summands=(
                OperatorSummand(replacement=(A_sum(),)),
                OperatorSummand(replacement=(B_sum(),)),
            )
        )
    if occ.field is eta_sum:
        return single_field_result((R_sum(),))
    return None


O_sum = FieldOperator(name="O_sum", parity=0, on_field=O_sum_slot)
L_sum_out = L_sum_slots.apply_operator(O_sum)
expr_sum_out = L_sum_out.to_symbolica()

print("Source:", sum_model.lagrangian_decl.source_terms[0])
print()
for i, term in enumerate(L_sum_out.terms):
    print("coupling", term.coupling)
    describe_term(term, i)
print()
print("Symbolica export:")
print(expr_sum_out)

assert len(L_sum_out.terms) == 3
assert [[field_factor(o) for o in term.fields] for term in L_sum_out.terms] == [
    ["A_sum", "eta_sum"],
    ["B_sum", "eta_sum"],
    ["phi_sum", "R_sum"],
]
assert expr_sum_out.expand().to_canonical_string() == Expression.parse(
    "g*A_sum*eta_sum + g*B_sum*eta_sum + g*phi_sum*R_sum"
).to_canonical_string()


Source: g * phi_sum * eta_sum

coupling g
[0] A_sum · eta_sum
     ∂: (none)   bilinears: (none)
coupling g
[1] B_sum · eta_sum
     ∂: (none)   bilinears: (none)
coupling g
[2] phi_sum · R_sum
     ∂: (none)   bilinears: (none)

Symbolica export:
g*phi_sum*R_sum+g*eta_sum*A_sum+g*eta_sum*B_sum


### 10.3 Derivative fan-out

If the operator commutes with partial derivatives, an existing derivative on a slot fans out across a product-valued replacement.


In [21]:
phi_df = scalar_field("phi_df", self_conjugate=True)
chi_df = scalar_field("chi_df", self_conjugate=True)
eta_df = scalar_field("eta_df", self_conjugate=True)

fan_model = Model(c * PartialD(phi_df, mu),
)
L_fan_slots = fan_model.lagrangian()
O_fan = replacement_operator(
    "O_fan",
    {phi_df: single_field_result((chi_df(), eta_df()))},
)
L_fan_out = L_fan_slots.apply_operator(O_fan)
expr_fan_out = L_fan_out.to_symbolica()
print("=======",expr_fan_out)
print("Source:", fan_model.lagrangian_decl.source_terms[0])
print()
for i, term in enumerate(L_fan_out.terms):
    print("coupling", term.coupling)
    describe_term(term, i)

print()
print("Feynman rules for (chi_df, eta_df):", L_fan_out.feynman_rule(chi_df, eta_df))

======= c*chi_df*PartialD(eta_df,mu)+c*eta_df*PartialD(chi_df,mu)
Source: c * PartialD(phi_df, mu)

coupling c
[0] chi_df · eta_df
     ∂: slot 0 → ∂_mu   bilinears: (none)
coupling c
[1] chi_df · eta_df
     ∂: slot 1 → ∂_mu   bilinears: (none)

Feynman rules for (chi_df, eta_df): c*pcomp(q1,mu1_int)+c*pcomp(q2,mu1_int)


## 11. Infinitesimal gauge variation: `gauge_variation`

The operator layer above is just *replacement plus graded Leibniz on lowered slots*. **Gauge transformations** are exactly this in disguise. For one gauge group, the **infinitesimal** variation acts as

* matter rep R: $\delta \Phi_i = + i g\, \alpha^a T^a_{ij}\, \Phi_j$, $\delta \bar\Phi_i = - i g\, \alpha^a \bar\Phi_j T^a_{ji}$
* abelian U(1) charge $q$: $\delta \Phi = + i g q\, \alpha\, \Phi$, $\delta \bar\Phi = - i g q\, \alpha\, \bar\Phi$
* gauge boson: $\delta A^a_\mu = +\partial_\mu \alpha^a - g f^{abc} \alpha^b A^c_\mu$ (non-abelian); $\delta A_\mu = +\partial_\mu \alpha$ (abelian)

`gauge_variation(group=..., parameter="alpha")` returns a `FieldOperator` we can hand to `L.apply_operator(...)`. Under the hood, $\alpha$ is materialised as a synthetic scalar field with the group's adjoint index. Repeated active non-abelian representation slots now expand in **first-seen slot order** when the representation uses `slot_policy="sum"`, and the acted slots still need explicit labels.


In [22]:
from model import CovD, Gamma, GaugeGroup, Field, LORENTZ_INDEX
from lagrangian.operator_action import gauge_variation
from symbolic.tensor_canonicalization import canonize_full

# A tiny U(1) example: charged complex scalar + charged Dirac fermion.
g_u1 = S("g")
A_u1 = Field("A", spin=1, indices=(LORENTZ_INDEX,), self_conjugate=True)
U1 = GaugeGroup(name="U1", abelian=True, coupling=g_u1, charge="Q", gauge_boson=A_u1)

Phi_q = scalar_field("Phi", self_conjugate=False, quantum_numbers={"Q": 1})
psi_q = dirac_field("psi", quantum_numbers={"Q": -1})

mu = S("mu")
m2 = S("m2")

L_u1 = Model(
    (
        m2 * Phi_q.bar * Phi_q
        + Expression.I * psi_q.bar() * Gamma(mu) * CovD(psi_q, mu)
    ),
    gauge_groups=(U1,),
).lagrangian()

delta_u1 = gauge_variation(group=U1, parameter="alpha")
dL_u1 = L_u1.apply_operator(delta_u1)

print("Symbolica view of delta(L):")
print(dL_u1.to_symbolica().expand())

Symbolica view of delta(L):
-g*gamma(bis(4, i_bar_psi_U1),bis(4, i_psi_U1),mink(4, mu))*psi(i_psi_U1)*psibar(i_bar_psi_U1)*PartialD(alpha,mu)+g*gamma(bis(4, i_bar_psi_covd),bis(4, i_psi_covd),mink(4, mu))*psi(i_psi_covd)*psibar(i_bar_psi_covd)*PartialD(alpha,mu)


The raw expression has spinor dummies introduced by the CovD lowering (`i_bar_psi_U1`, `i_psi_covd`, ...). The vertex canonicaliser used in the rest of the pipeline reduces this to **`0`**:

In [23]:
spinor_dummies = (
    S("i_bar_psi_U1"), S("i_psi_U1"),
    S("i_bar_psi_covd"), S("i_psi_covd"),
)
canonical = canonize_full(
    dL_u1.to_symbolica(),
    lorentz_indices=(mu,),
    spinor_indices=spinor_dummies,
)
print("delta(L) canonicalised:")
print(canonical)
assert canonical.to_canonical_string() == "0"
print("\n=> L = m^2 Phibar Phi + i Psibar gamma^mu D_mu Psi is gauge invariant.")

delta(L) canonicalised:
0

=> L = m^2 Phibar Phi + i Psibar gamma^mu D_mu Psi is gauge invariant.


### 11.1 Wrong-charge sanity check

If we couple a charged scalar to a neutral Yukawa pair (so total $U(1)$ charge is non-zero), the variation should **not** vanish — it pinpoints exactly the charge-conservation violation.

In [24]:
psi_neutral = dirac_field("eta", quantum_numbers={"Q": 0})
L_wrong = Model(S("y") * psi_neutral.bar() * psi_neutral() * Phi_q).lagrangian()

dL_wrong = L_wrong.apply_operator(delta_u1)
print("delta(y * eta.bar * eta * Phi)  (Phi has Q=1, eta has Q=0):")
print(dL_wrong.to_symbolica().expand())
print("\n=> non-zero: charge is not conserved by this Yukawa.")

delta(y * eta.bar * eta * Phi)  (Phi has Q=1, eta has Q=0):
1𝑖*Phi*g*alpha*y*eta(i_decl_1)*etabar(i_decl_1)

=> non-zero: charge is not conserved by this Yukawa.


In [25]:
L_charged = Model(S("y") * psi_q.bar() * psi_q() * Phi).lagrangian()
dL_charged = L_charged.apply_operator(delta_u1)
print("delta(y * psi.bar * psi * Phi)  (Phi has Q=1, psi has Q=-1):")
print(dL_charged.to_symbolica().expand())
print("\n=> zero: charge is conserved by this Yukawa.")

delta(y * psi.bar * psi * Phi)  (Phi has Q=1, psi has Q=-1):
0

=> zero: charge is conserved by this Yukawa.


### 11.2 Non-abelian gauge-boson variation

For a non-abelian group the gauge-boson variation has two pieces: an **inhomogeneous** $+\partial_\mu \alpha^a$ and a **homogeneous** $-g f^{abc}\alpha^b A^c_\mu$. We can apply `gauge_variation` to a single $A^a_\mu$ slot to see them produced explicitly.

In [26]:
from model.metadata import COLOR_ADJ_INDEX, COLOR_FUND_INDEX
from model import GaugeRepresentation
from symbolic.spenso_structures import gauge_generator, structure_constant

g_s = S("gS")
G_field =Field(
    "G",
    spin=1,
    indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX),
    self_conjugate=True,
)
SU3 = GaugeGroup(
    name="SU3",
    abelian=False,
    coupling=g_s,
    gauge_boson=G_field,
    structure_constant=structure_constant,
    representations=(
        GaugeRepresentation(
            index=COLOR_FUND_INDEX,
            generator_builder=gauge_generator,
            name="fundamental",
        ),
    ),
)

mu_na = S("mu")
aa = S("aa")
L_A = Model(
    G_field(mu_na, aa),
    gauge_groups=(SU3,),
).lagrangian()

delta_su3 = gauge_variation(group=SU3, parameter="alpha")
dL_A = L_A.apply_operator(delta_su3)
print("delta(A^a_mu):")
print(dL_A.to_symbolica().expand())

delta(A^a_mu):
-gS*f(coad(8, aa),coad(8, a_delta_SU3_1),coad(8, a_delta_SU3_2))*alpha(a_delta_SU3_1)*G(mu,a_delta_SU3_2)+PartialD(alpha(aa),mu)


### 11.3 Repeated representation slots (`slot_policy="sum"`)

For repeated non-abelian matter slots, `gauge_variation(...)` now emits one contribution per active slot, in the same order as the covariant compiler.


In [27]:
Phi_bi = scalar_field(
    "PhiBi",
    self_conjugate=False,
    indices=(COLOR_FUND_INDEX, COLOR_FUND_INDEX),
)
SU3_sum = GaugeGroup(
    name="SU3_sum",
    abelian=False,
    coupling=g_s,
    gauge_boson=G_field,
    structure_constant=structure_constant,
    representations=(
        GaugeRepresentation(
            index=COLOR_FUND_INDEX,
            generator_builder=gauge_generator,
            name="fundamental_sum",
            slot_policy="sum",
        ),
    ),
)

c1, c2 = S("c1", "c2")
L_bi = Model(
    Phi_bi(c1, c2),
    gauge_groups=(SU3_sum,),
).lagrangian()

delta_sum = gauge_variation(group=SU3_sum, parameter="alpha_sum")
print("slot_policy='sum' matter variation:")
for i, term in enumerate(L_bi.apply_operator(delta_sum).terms):
    describe_term(term, i)
print("deltaL =", L_bi.apply_operator(delta_sum).to_symbolica())
try:
    bad_occ = Phi_bi.occurrence(labels={COLOR_FUND_INDEX.kind: (c1, None)})
    delta_sum(bad_occ)
except ValueError as exc:
    print("missing-label failure:", exc)


slot_policy='sum' matter variation:
[0] alpha_sum · PhiBi
     ∂: (none)   bilinears: (none)
[1] alpha_sum · PhiBi
     ∂: (none)   bilinears: (none)
deltaL = 1𝑖*gS*t(coad(8, a_delta_SU3_sum_6),cof(3, c1),cof(3, c_delta_SU3_sum_5))*PhiBi(c_delta_SU3_sum_5,c2)*alpha_sum(a_delta_SU3_sum_6)+1𝑖*gS*t(coad(8, a_delta_SU3_sum_8),cof(3, c2),cof(3, c_delta_SU3_sum_7))*PhiBi(c1,c_delta_SU3_sum_7)*alpha_sum(a_delta_SU3_sum_8)
missing-label failure: gauge_variation('SU3_sum'): field 'PhiBi' has no explicit label on representation slot 2; cannot insert a contracted generator without one.


$\delta \mathcal{L} = i g_s \alpha^a
\left[
t^a_{c_1 d},\Phi_{\mathrm{Bi}}(d,c_2)
+
t^a_{c_2 d},\Phi_{\mathrm{Bi}}(c_1,d)
\right]$

$-\frac14 F^a_{\mu\nu} F^{a\mu\nu}.$



In [ ]:
from model import FieldStrength

mu_fs = S("mu_fs")
nu_fs = S("nu_fs")
L_su3_ym = Model(
    gauge_groups=(SU3,),
    fields=(G_field,),
    lagrangian_decl=-(Expression.num(1) / Expression.num(4))
    * FieldStrength(SU3, mu_fs, nu_fs)
    * FieldStrength(SU3, mu_fs, nu_fs),
).lagrangian()

dL_su3_ym = L_su3_ym.apply_operator(delta_su3)
expr_su3_ym = dL_su3_ym.to_symbolica().expand()
canon_su3_ym = canonize_full(expr_su3_ym, lorentz_indices=(mu_fs, nu_fs))

raw_text = expr_su3_ym.to_canonical_string()
canon_text = canon_su3_ym.to_canonical_string()

print("Pure SU(3) Yang-Mills variation: -1/4 F^2")
print("number of compiled variation terms:", len(dL_su3_ym.terms))
print("raw canonical-string length:", len(raw_text))
print("after canonize_full(...):", len(canon_text))
print("canonicalized to zero?", canon_text == "0")
print("=> with the default color simplification on, this pass does not stay in the explicit f-basis.")


In [29]:
print("lagrangian:", L_su3_ym.to_symbolica())
print("delta(L):", dL_su3_ym.to_symbolica())

lagrangian: -gS*g(mink(4, mu_G_SU3_1),mink(4, mu_G_SU3_3))*g(mink(4, mu_G_SU3_2),mink(4, rho_G_SU3_cubic))*f(coad(8, color_adj_G_SU3_1),coad(8, color_adj_G_SU3_2),coad(8, color_adj_G_SU3_3))*PartialD(G(mu_G_SU3_1,color_adj_G_SU3_1),rho_G_SU3_cubic)*G(mu_G_SU3_2,color_adj_G_SU3_2)*G(mu_G_SU3_3,color_adj_G_SU3_3)-1/4*gS^2*g(mink(4, mu_G_SU3_1),mink(4, mu_G_SU3_3))*g(mink(4, mu_G_SU3_2),mink(4, mu_G_SU3_4))*f(coad(8, color_adj_G_SU3_1),coad(8, color_adj_G_SU3_2),coad(8, a_mid_G_SU3))*f(coad(8, color_adj_G_SU3_3),coad(8, color_adj_G_SU3_4),coad(8, a_mid_G_SU3))*G(mu_G_SU3_1,color_adj_G_SU3_1)*G(mu_G_SU3_2,color_adj_G_SU3_2)*G(mu_G_SU3_3,color_adj_G_SU3_3)*G(mu_G_SU3_4,color_adj_G_SU3_4)-1/2*g(mink(4, mu_G_SU3_1),mink(4, mu_G_SU3_2))*g(coad(8, a_bar_G_color_adj_id),coad(8, a_G_color_adj_id))*PartialD(G(mu_G_SU3_1,a_bar_G_color_adj_id),rho_G_SU3)*PartialD(G(mu_G_SU3_2,a_G_color_adj_id),rho_G_SU3)+1/2*g(mink(4, mu_G_SU3_1),mink(4, rho_right_G_SU3))*g(mink(4, mu_G_SU3_2),mink(4, rho_left_G_SU3


$
\eta_{\mu\nu}=g(\text{mink}(4,\mu),\text{mink}(4,\nu)),
\qquad
\delta_{ab}=g(\text{coad}(8,a),\text{coad}(8,b)).
$

$
\begin{aligned}
\mathcal L
={}&
-g_s,
\eta_{\mu_1\mu_3}\eta_{\mu_2\rho},
f^{a_1a_2a_3},
(\partial_\rho G_{\mu_1}^{a_1})
G_{\mu_2}^{a_2}
G_{\mu_3}^{a_3}
[2mm]
&-\frac14 g_s^2,
\eta_{\mu_1\mu_3}\eta_{\mu_2\mu_4},
f^{a_1a_2b}f^{a_3a_4b},
G_{\mu_1}^{a_1}
G_{\mu_2}^{a_2}
G_{\mu_3}^{a_3}
G_{\mu_4}^{a_4}
[2mm]
&-\frac12,
\eta_{\mu_1\mu_2}\delta_{ab},
(\partial_\rho G_{\mu_1}^{a})
(\partial_\rho G_{\mu_2}^{b})
[2mm]
&+\frac12,
\eta_{\mu_1\rho_R}\eta_{\mu_2\rho_L}\delta_{ab},
(\partial_{\rho_L}G_{\mu_1}^{a})
(\partial_{\rho_R}G_{\mu_2}^{b}) .
\end{aligned}
$


$
\begin{aligned}
\delta\mathcal L
={}&
\frac12 g_s,
\eta_{\mu_1\mu_2}\delta_{ab}
f^{a c d},
(\partial_\rho \alpha^c)
(\partial_\rho G_{\mu_2}^{b})
G_{\mu_1}^{d}
\
&+\frac12 g_s,
\eta_{\mu_1\mu_2}\delta_{ab}
f^{a c d},
(\partial_\rho G_{\mu_1}^{d})
(\partial_\rho G_{\mu_2}^{b})
\alpha^c
\
&+\frac12 g_s,
\eta_{\mu_1\mu_2}\delta_{ab}
f^{b c d},
(\partial_\rho \alpha^c)
(\partial_\rho G_{\mu_1}^{a})
G_{\mu_2}^{d}
\
&+\frac12 g_s,
\eta_{\mu_1\mu_2}\delta_{ab}
f^{b c d},
(\partial_\rho G_{\mu_1}^{a})
(\partial_\rho G_{\mu_2}^{d})
\alpha^c
[2mm]
&-\frac12 g_s,
\eta_{\mu_1\rho_R}\eta_{\mu_2\rho_L}\delta_{ab}
f^{a c d},
(\partial_{\rho_L}\alpha^c)
(\partial_{\rho_R}G_{\mu_2}^{b})
G_{\mu_1}^{d}
\
&-\frac12 g_s,
\eta_{\mu_1\rho_R}\eta_{\mu_2\rho_L}\delta_{ab}
f^{a c d},
(\partial_{\rho_L}G_{\mu_1}^{d})
(\partial_{\rho_R}G_{\mu_2}^{b})
\alpha^c
\
&-\frac12 g_s,
\eta_{\mu_1\rho_R}\eta_{\mu_2\rho_L}\delta_{ab}
f^{b c d},
(\partial_{\rho_R}\alpha^c)
(\partial_{\rho_L}G_{\mu_1}^{a})
G_{\mu_2}^{d}
\
&-\frac12 g_s,
\eta_{\mu_1\rho_R}\eta_{\mu_2\rho_L}\delta_{ab}
f^{b c d},
(\partial_{\rho_L}G_{\mu_1}^{a})
(\partial_{\rho_R}G_{\mu_2}^{d})
\alpha^c
[2mm]
&-g_s,
\eta_{\mu_1\mu_3}\eta_{\mu_2\rho}
f^{a_1a_2a_3}
(\partial_{\mu_2}\alpha^{a_2})
(\partial_\rho G_{\mu_1}^{a_1})
G_{\mu_3}^{a_3}
\
&-g_s,
\eta_{\mu_1\mu_3}\eta_{\mu_2\rho}
f^{a_1a_2a_3}
(\partial_{\mu_3}\alpha^{a_3})
(\partial_\rho G_{\mu_1}^{a_1})
G_{\mu_2}^{a_2}
\
&-g_s,
\eta_{\mu_1\mu_3}\eta_{\mu_2\rho}
f^{a_1a_2a_3}
(\partial_{\mu_1}\partial_\rho\alpha^{a_1})
G_{\mu_2}^{a_2}
G_{\mu_3}^{a_3}
[2mm]
&+g_s^2,
\eta_{\mu_1\mu_3}\eta_{\mu_2\rho}
f^{a_1a_2a_3}f^{a_1cd}
(\partial_\rho\alpha^c)
G_{\mu_1}^{d}
G_{\mu_2}^{a_2}
G_{\mu_3}^{a_3}
\
&+g_s^2,
\eta_{\mu_1\mu_3}\eta_{\mu_2\rho}
f^{a_1a_2a_3}f^{a_1cd}
(\partial_\rho G_{\mu_1}^{d})
\alpha^c
G_{\mu_2}^{a_2}
G_{\mu_3}^{a_3}
\
&+g_s^2,
\eta_{\mu_1\mu_3}\eta_{\mu_2\rho}
f^{a_1a_2a_3}f^{a_2cd}
(\partial_\rho G_{\mu_1}^{a_1})
\alpha^c
G_{\mu_2}^{d}
G_{\mu_3}^{a_3}
\
&+g_s^2,
\eta_{\mu_1\mu_3}\eta_{\mu_2\rho}
f^{a_1a_2a_3}f^{a_3cd}
(\partial_\rho G_{\mu_1}^{a_1})
\alpha^c
G_{\mu_2}^{a_2}
G_{\mu_3}^{d}
[2mm]
&-\frac14 g_s^2,
\eta_{\mu_1\mu_3}\eta_{\mu_2\mu_4}
f^{a_1a_2b}f^{a_3a_4b}
(\partial_{\mu_1}\alpha^{a_1})
G_{\mu_2}^{a_2}
G_{\mu_3}^{a_3}
G_{\mu_4}^{a_4}
\
&-\frac14 g_s^2,
\eta_{\mu_1\mu_3}\eta_{\mu_2\mu_4}
f^{a_1a_2b}f^{a_3a_4b}
(\partial_{\mu_2}\alpha^{a_2})
G_{\mu_1}^{a_1}
G_{\mu_3}^{a_3}
G_{\mu_4}^{a_4}
\
&-\frac14 g_s^2,
\eta_{\mu_1\mu_3}\eta_{\mu_2\mu_4}
f^{a_1a_2b}f^{a_3a_4b}
(\partial_{\mu_3}\alpha^{a_3})
G_{\mu_1}^{a_1}
G_{\mu_2}^{a_2}
G_{\mu_4}^{a_4}
\
&-\frac14 g_s^2,
\eta_{\mu_1\mu_3}\eta_{\mu_2\mu_4}
f^{a_1a_2b}f^{a_3a_4b}
(\partial_{\mu_4}\alpha^{a_4})
G_{\mu_1}^{a_1}
G_{\mu_2}^{a_2}
G_{\mu_3}^{a_3}
[2mm]
&+\frac14 g_s^3,
\eta_{\mu_1\mu_3}\eta_{\mu_2\mu_4}
f^{a_1a_2b}f^{a_1cd}f^{a_3a_4b}
\alpha^c
G_{\mu_1}^{d}
G_{\mu_2}^{a_2}
G_{\mu_3}^{a_3}
G_{\mu_4}^{a_4}
\
&+\frac14 g_s^3,
\eta_{\mu_1\mu_3}\eta_{\mu_2\mu_4}
f^{a_1a_2b}f^{a_2cd}f^{a_3a_4b}
\alpha^c
G_{\mu_1}^{a_1}
G_{\mu_2}^{d}
G_{\mu_3}^{a_3}
G_{\mu_4}^{a_4}
\
&+\frac14 g_s^3,
\eta_{\mu_1\mu_3}\eta_{\mu_2\mu_4}
f^{a_1a_2b}f^{a_3a_4b}f^{a_3cd}
\alpha^c
G_{\mu_1}^{a_1}
G_{\mu_2}^{a_2}
G_{\mu_3}^{d}
G_{\mu_4}^{a_4}
\
&+\frac14 g_s^3,
\eta_{\mu_1\mu_3}\eta_{\mu_2\mu_4}
f^{a_1a_2b}f^{a_3a_4b}f^{a_4cd}
\alpha^c
G_{\mu_1}^{a_1}
G_{\mu_2}^{a_2}
G_{\mu_3}^{a_3}
G_{\mu_4}^{d}
[2mm]
&-\frac12
\eta_{\mu_1\mu_2}\delta_{ab}
(\partial_\rho G_{\mu_1}^{a})
(\partial_{\mu_2}\partial_\rho\alpha^b)
\
&-\frac12
\eta_{\mu_1\mu_2}\delta_{ab}
(\partial_\rho G_{\mu_2}^{b})
(\partial_{\mu_1}\partial_\rho\alpha^a)
\
&+\frac12
\eta_{\mu_1\rho_R}\eta_{\mu_2\rho_L}\delta_{ab}
(\partial_{\rho_L}G_{\mu_1}^{a})
(\partial_{\mu_2}\partial_{\rho_R}\alpha^b)
\
&+\frac12
\eta_{\mu_1\rho_R}\eta_{\mu_2\rho_L}\delta_{ab}
(\partial_{\rho_R}G_{\mu_2}^{b})
(\partial_{\mu_1}\partial_{\rho_L}\alpha^a).
\end{aligned}
$


To use step 6 cleanly we keep the color algebra in the `f` basis with `run_color=False` and let `canonize_full(..., infer_indices=True)` collect the Lorentz and adjoint symbols from the exported heads (`G`, `alpha`, `PartialD`, `f`). The direct Jacobi identity now collapses to zero, and after adding commuting-`PartialD` canonicalization plus antisymmetric zero-term elimination, the same pipeline sends the full `\delta L_{\mathrm{SU(3)\,YM}}` to zero.


In [ ]:
a_j, b_j, c_j, d_j, e_j = S("a_j", "b_j", "c_j", "d_j", "e_j")

jacobi_expr = (
    structure_constant(a_j, b_j, e_j) * structure_constant(c_j, d_j, e_j)
    - structure_constant(a_j, c_j, e_j) * structure_constant(b_j, d_j, e_j)
    + structure_constant(a_j, d_j, e_j) * structure_constant(b_j, c_j, e_j)
)

jacobi_canon = canonize_full(
    jacobi_expr,
    adjoint_indices=(a_j, b_j, c_j, d_j, e_j),
    run_gamma=False,
    run_color=False,
)

print("Jacobi identity in the f-basis:")
print(jacobi_canon)


In [ ]:
canon_su3_ym_jacobi = canonize_full(
    expr_su3_ym,
    run_gamma=False,
    run_color=False,
    infer_indices=True,
)
canon_su3_ym_jacobi_text = canon_su3_ym_jacobi.to_canonical_string()

print("Final SU(3) Yang-Mills simplification in the f-basis:")
print("raw canonical-string length:", len(raw_text))
print("after canonize_full(..., run_color=False):", len(canon_su3_ym_jacobi_text))
print("canonicalized to zero?", canon_su3_ym_jacobi_text == "0")
print(canon_su3_ym_jacobi)


### 11.4 Higher field-strength operator across several gauge symmetries

The Jacobi-based reduction is not limited to the renormalizable `-1/4 F^2` term. We can also build a mixed pure-gauge stack with three symmetry groups and a higher operator $$\kappa\,(G^a_{\mu\nu}G^{a\mu\nu})(W^i_{\rho\sigma}W^{i\rho\sigma})$$ and check that the infinitesimal `SU(3)` and `SU(2)` variations still canonicalize to zero.


In [ ]:
from model import WEAK_ADJ_INDEX, WEAK_FUND_INDEX
from symbolic.spenso_structures import weak_gauge_generator, weak_structure_constant

g2 = S("g2")
g1 = S("g1")
kappa = S("kappa")

W_field = Field(
    "W",
    spin=1,
    indices=(LORENTZ_INDEX, WEAK_ADJ_INDEX),
    self_conjugate=True,
)
B_field = Field("B", spin=1, indices=(LORENTZ_INDEX,), self_conjugate=True)

SU2L = GaugeGroup(
    name="SU2L",
    abelian=False,
    coupling=g2,
    gauge_boson=W_field,
    structure_constant=weak_structure_constant,
    representations=(
        GaugeRepresentation(
            index=WEAK_FUND_INDEX,
            generator_builder=weak_gauge_generator,
            name="doublet",
        ),
    ),
)
U1Y = GaugeGroup(name="U1Y", abelian=True, coupling=g1, gauge_boson=B_field, charge="Y")

mu_w = S("mu_w")
nu_w = S("nu_w")
mu_b = S("mu_b")
nu_b = S("nu_b")

L_mixed_fs = Model(
    gauge_groups=(SU3, SU2L, U1Y),
    fields=(G_field, W_field, B_field),
    lagrangian_decl=(
        -(Expression.num(1) / Expression.num(4))
        * FieldStrength(SU3, mu_fs, nu_fs)
        * FieldStrength(SU3, mu_fs, nu_fs)
        - (Expression.num(1) / Expression.num(4))
        * FieldStrength(SU2L, mu_w, nu_w)
        * FieldStrength(SU2L, mu_w, nu_w)
        - (Expression.num(1) / Expression.num(4))
        * FieldStrength(U1Y, mu_b, nu_b)
        * FieldStrength(U1Y, mu_b, nu_b)
        + kappa
        * FieldStrength(SU3, mu_fs, nu_fs)
        * FieldStrength(SU3, mu_fs, nu_fs)
        * FieldStrength(SU2L, mu_w, nu_w)
        * FieldStrength(SU2L, mu_w, nu_w)
    ),
).lagrangian()

print("compiled interaction terms in the mixed gauge stack:", len(L_mixed_fs.terms))
for group in (SU3, SU2L):
    delta_group = gauge_variation(group=group, parameter="alpha")
    dL_group = L_mixed_fs.apply_operator(delta_group)
    expr_group = dL_group.to_symbolica().expand()
    canon_group = canonize_full(
        expr_group,
        run_gamma=False,
        run_color=False,
        infer_indices=True,
    )
    print()
    print(f"Gauge variation for {group.name}:")
    print("  variation terms:", len(dL_group.terms))
    print("  canonicalized to zero?", canon_group.to_canonical_string() == "0")


In [30]:
expr = L_su3_ym.to_symbolica()
G = G_field.species_for()
mu1_, a1_, mu2_, a2_ = S("mu1_", "a1_", "mu2_", "a2_")
pattern = G(mu1_, a1_) * G(mu2_, a2_)
print("Lowered Yang-Mills export:")
print(expr)
print("Literal coefficient(...) with wildcard symbols stays literal in Symbolica:")
print(expr.coefficient(pattern))
print("Wildcard-aware pattern coefficient on the compiled lagrangian:")
print(L_su3_ym.pattern_coefficient(pattern))

Lowered Yang-Mills export:
-gS*g(mink(4, mu_G_SU3_1),mink(4, mu_G_SU3_3))*g(mink(4, mu_G_SU3_2),mink(4, rho_G_SU3_cubic))*f(coad(8, color_adj_G_SU3_1),coad(8, color_adj_G_SU3_2),coad(8, color_adj_G_SU3_3))*PartialD(G(mu_G_SU3_1,color_adj_G_SU3_1),rho_G_SU3_cubic)*G(mu_G_SU3_2,color_adj_G_SU3_2)*G(mu_G_SU3_3,color_adj_G_SU3_3)-1/4*gS^2*g(mink(4, mu_G_SU3_1),mink(4, mu_G_SU3_3))*g(mink(4, mu_G_SU3_2),mink(4, mu_G_SU3_4))*f(coad(8, color_adj_G_SU3_1),coad(8, color_adj_G_SU3_2),coad(8, a_mid_G_SU3))*f(coad(8, color_adj_G_SU3_3),coad(8, color_adj_G_SU3_4),coad(8, a_mid_G_SU3))*G(mu_G_SU3_1,color_adj_G_SU3_1)*G(mu_G_SU3_2,color_adj_G_SU3_2)*G(mu_G_SU3_3,color_adj_G_SU3_3)*G(mu_G_SU3_4,color_adj_G_SU3_4)-1/2*g(mink(4, mu_G_SU3_1),mink(4, mu_G_SU3_2))*g(coad(8, a_bar_G_color_adj_id),coad(8, a_G_color_adj_id))*PartialD(G(mu_G_SU3_1,a_bar_G_color_adj_id),rho_G_SU3)*PartialD(G(mu_G_SU3_2,a_G_color_adj_id),rho_G_SU3)+1/2*g(mink(4, mu_G_SU3_1),mink(4, rho_right_G_SU3))*g(mink(4, mu_G_SU3_2),mink(4,

In [31]:
expr = L_su3_ym.to_symbolica()
G = S("G")
mu1_, a1_, mu2_, a2_ = S("mu1_", "a1_", "mu2_", "a2_")
pattern = G(mu1_, a1_) * G(mu2_, a2_)

print("Using only Symbolica built-ins: match + replace_wildcards + coefficient")
print("literal coefficient(pattern):", expr.coefficient(pattern))

seen = {}
for match in expr.match(pattern, min_level=0, max_level=0, partial=True):
    factor = pattern.replace_wildcards(match)
    seen.setdefault(factor.to_canonical_string(), factor)

coeff_total = Expression.num(0)
for factor in seen.values():
    coeff = expr.coefficient(factor)
    coeff_total += coeff
    print("factor:", factor)
    print("coefficient:", coeff)

print("sum of unique matched coefficients:")
print(coeff_total)


Using only Symbolica built-ins: match + replace_wildcards + coefficient
literal coefficient(pattern): 0
factor: G(mu_G_SU3_2,color_adj_G_SU3_2)*G(mu_G_SU3_3,color_adj_G_SU3_3)
coefficient: -gS*g(mink(4, mu_G_SU3_1),mink(4, mu_G_SU3_3))*g(mink(4, mu_G_SU3_2),mink(4, rho_G_SU3_cubic))*f(coad(8, color_adj_G_SU3_1),coad(8, color_adj_G_SU3_2),coad(8, color_adj_G_SU3_3))*PartialD(G(mu_G_SU3_1,color_adj_G_SU3_1),rho_G_SU3_cubic)-1/4*gS^2*g(mink(4, mu_G_SU3_1),mink(4, mu_G_SU3_3))*g(mink(4, mu_G_SU3_2),mink(4, mu_G_SU3_4))*f(coad(8, color_adj_G_SU3_1),coad(8, color_adj_G_SU3_2),coad(8, a_mid_G_SU3))*f(coad(8, color_adj_G_SU3_3),coad(8, color_adj_G_SU3_4),coad(8, a_mid_G_SU3))*G(mu_G_SU3_1,color_adj_G_SU3_1)*G(mu_G_SU3_4,color_adj_G_SU3_4)
factor: G(mu_G_SU3_1,color_adj_G_SU3_1)*G(mu_G_SU3_2,color_adj_G_SU3_2)
coefficient: -1/4*gS^2*g(mink(4, mu_G_SU3_1),mink(4, mu_G_SU3_3))*g(mink(4, mu_G_SU3_2),mink(4, mu_G_SU3_4))*f(coad(8, color_adj_G_SU3_1),coad(8, color_adj_G_SU3_2),coad(8, a_mid_G_SU3))*